# 🤖 Notebook 4: Model Training

# 04 Model Training
This notebook implements the training wrappers for all four forecasting models: SARIMA, Prophet, XGBoost, and LSTM.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

## Training vs Validation Split Strategy

### Why We Never Shuffle Time-Series Data
In standard machine learning, we often shuffle data to ensure the model doesn't learn the order of samples. However, in time-series forecasting, **we NEVER shuffle the data**. Shuffling would allow the model to 'see' the future to predict the past (data leakage), breaking the temporal dependency. Instead, we use a chronological split (Time Series Split) where the training set contains the early period and the validation set contains the most recent contiguous period.

In [ ]:
def time_series_split(df, val_weeks=8):
    df = df.sort_values('Date')
    unique_dates = sorted(df['Date'].unique())
    split_date = unique_dates[-val_weeks]
    train_df = df[df['Date'] < split_date]
    val_df = df[df['Date'] >= split_date]
    return train_df, val_df

print("Temporal splitting logic defined.")

## Model 1: SARIMA

A traditional statistics-based model that finds patterns by looking at past values (autoregression), correcting for trends (integration), and smoothing errors (moving average) — all while accounting for repeating seasonal cycles.

In [ ]:
class SARIMAForecaster:
    def __init__(self, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0)):
        self.order = order
        self.seasonal_order = seasonal_order
        self.model_result = None

    def fit(self, series):
        model = SARIMAX(series, order=self.order, seasonal_order=self.seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
        self.model_result = model.fit(disp=False)
        return self.model_result

    def predict(self, steps):
        forecast = self.model_result.get_forecast(steps=steps)
        return forecast.predicted_mean

## Model 2: Prophet

Facebook's easy-to-use forecasting tool that automatically detects weekly and yearly patterns, adjusts for holidays, and handles sudden trend changes — designed specifically for messy, real-world business data.

In [ ]:
class ProphetForecaster:
    def __init__(self):
        self.model = Prophet(weekly_seasonality=True, yearly_seasonality=True)
        self.is_fitted = False

    def fit(self, series):
        df = series.reset_index()
        df.columns = ['ds', 'y']
        self.model.fit(df)
        self.is_fitted = True

    def predict(self, steps):
        future = self.model.make_future_dataframe(periods=steps, freq='W', include_history=False)
        forecast = self.model.predict(future)
        return forecast[['ds', 'yhat']].set_index('ds')['yhat']

## Model 3: XGBoost

A powerful machine learning model that learns from our engineered features (lags, rolling averages, holidays) by building many small decision trees that each correct the errors of the previous one.

In [ ]:
class XGBoostForecaster:
    def __init__(self, n_estimators=100):
        self.model = xgb.XGBRegressor(n_estimators=n_estimators, objective='reg:squarederror')
        self.is_fitted = False

    def fit(self, X, y):
        self.model.fit(X, y)
        self.is_fitted = True

    def predict(self, X):
        return self.model.predict(X)

## Model 4: LSTM

A type of neural network with built-in memory that reads sequences of past sales week by week, remembers important long-range patterns, and uses them to predict what comes next.

In [ ]:
class LSTMForecaster:
    def __init__(self, n_steps=8, n_features=1):
        self.n_steps = n_steps
        self.n_features = n_features
        self.model = Sequential([
            LSTM(50, activation='relu', input_shape=(n_steps, n_features)),
            Dense(1)
        ])
        self.model.compile(optimizer='adam', loss='mse')
        self.scaler = MinMaxScaler()

    def fit(self, series, epochs=10):
        values = self.scaler.fit_transform(series.values.reshape(-1, 1)).flatten()
        X, y = [], []
        for i in range(len(values)):
            end_ix = i + self.n_steps
            if end_ix > len(values) - 1: break
            X.append(values[i:end_ix])
            y.append(values[end_ix])
        X = np.array(X).reshape((-1, self.n_steps, self.n_features))
        self.model.fit(X, np.array(y), epochs=epochs, verbose=0)

    def predict(self, series, n_future_steps):
        return [0] * n_future_steps # Placeholder for actual recursive logic